# Combine sites

## In short

This notebook joins the site master CSVs (one for each site) into one table for all sites. Use it after `1_merge_dates_by_site.ipynb`.

To use it, list the site master CSVs and set the output folder at the top of the code cell. Then run the cell.

## Details

**Input**

The site master CSVs from `1_merge_dates_by_site.ipynb`. All files must have the same image type (all MS or all RGB).

**Output**

`output_folder/<image type>_MasterDataset_AllSites.csv`

**Column order**

The first file in `master_files` sets the column order. If another file has different columns, the notebook prints the differences. Then it changes that file to match the first file:

- A column that is not in the file is added with empty values.
- A column that is not in the first file is removed.

Put first the file that has the columns that you want to keep.

**How to run**

1. Set `image_type_label` to `"MS"` or `"RGB"`.
2. Add the path of each site master CSV to `master_files`.
3. Set `output_folder`.
4. Run the cell.
5. Check the printout. Make sure that each file shows `columns match`, or that the column differences are correct.

**Notes**

- Do not combine RGB files and MS files in one run.
- Rows with an empty plot ID (`var`) stay in the output. These rows come from grid polygons that have no plot ID. You can remove them after the merge if necessary.
- Empty values are written as `nan`.


In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# Combine site master datasets into one all-sites master dataset
# ============================================================

# ============================================================
# USER SETTINGS
# ============================================================

# "MS" or "RGB". Used in the output file name.
image_type_label = "MS"

# 1. Site master CSVs from 1_merge_dates_by_site.
# The first file sets the column order.
master_files = [
    Path('path/to/SiteA_MSorRGB_master.csv'),
    Path('path/to/SiteB_MSorRGB_master.csv'),
]

# The all-sites master CSV is written here.
output_folder = Path('path/to/master_output')

# 2. Check that every file exists before reading
print("Checking files...")
for file in master_files:
    print(file.name, "exists:", file.exists())

# 3. Read each master file
all_data = []

for file in master_files:
    print("-" * 70)
    print("Reading:", file.name)

    df = pd.read_csv(file)

    print("Shape:", df.shape)
    print("Columns:", len(df.columns))

    all_data.append(df)

print("-" * 70)
print("Total master files loaded:", len(all_data))

# 4. Use the first file as the reference column format
reference_columns = list(all_data[0].columns)

print("\nReference column order comes from:")
print(master_files[0].name)

# 5. Check column consistency and force same column order
for i, df in enumerate(all_data):
    current_columns = list(df.columns)

    if current_columns == reference_columns:
        print(f"File {i}: columns match")
    else:
        print(f"File {i}: columns do NOT perfectly match")
        print("File:", master_files[i].name)

        missing_cols = set(reference_columns) - set(current_columns)
        extra_cols = set(current_columns) - set(reference_columns)

        print("Missing columns:", missing_cols)
        print("Extra columns:", extra_cols)

        all_data[i] = df.reindex(columns=reference_columns)

# 6. Combine all MS datasets
master_df = pd.concat(all_data, ignore_index=True)

print(f"\nCombined {image_type_label} master dataset created.")
print("Combined shape:", master_df.shape)

# 7. Quality checks
print("\nRows by site:")
display(master_df["Site"].value_counts())

print("\nRows by date:")
display(master_df["Date"].value_counts().sort_index())

if "Missing_val" in master_df.columns:
    print("\nMissing values by site:")
    display(
        master_df
        .groupby("Site")["Missing_val"]
        .sum()
        .sort_values(ascending=False)
    )

    print("\nTotal missing values:")
    display(master_df["Missing_val"].sum())

# 8. Preview
display(master_df.head())
display(master_df.tail())

# 9. Save the all-sites master dataset
output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / f"{image_type_label}_MasterDataset_AllSites.csv"

master_df.to_csv(output_file, index=False, na_rep="nan")

print(f"\nSaved combined {image_type_label} master dataset to:")
print(output_file)